# Develop acceptance loop

In [1]:
from sklearn.ensemble import IsolationForest
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataGenerator, CreditData, CreditDataSample
from berebasl.simulation.acceptance_loop import accept_based_on_top_percentent_of_arbitrary_var
from berebasl.estimation.basl import BASLPartialUnbiaser
from berebasl.estimation.bayesian_evaluation import BayesianMetric, batched_auroc
from berebasl.estimation.classifiers import TorchLogistic

In [ ]:
from typing import Any, Dict, List, Union
dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else 
    "cpu"
)

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 200
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0], dtype=dtype, device=device),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]], dtype=dtype, device=device),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]], dtype=dtype, device=device)
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    dtype=dtype
)

# Initial population
data_gen.manual_seed(initial_seed)
feats_new_applicants, def_flag_new_applicants = data_gen.sample(init_sample, determinstic_mixture_weights)

accepts = accept_based_on_top_percentent_of_arbitrary_var(
    feats_new_applicants, 
    def_flag_new_applicants,
    var_for_rule=0,
    top_percent=top_percent,
    default_value = CreditDataGenerator.bad_good_encoding["bad"]
)

credit_data = CreditData(feats_new_applicants, def_flag_new_applicants, accepts)

# Holdout Population
data_gen.manual_seed(-initial_seed)
holdout_features, holdout_flag = data_gen.sample(n=holdout_sample)
holdout_data = CreditData(
    holdout_features, holdout_flag, 
    accepted_initial=torch.ones(holdout_flag.shape, dtype=torch.bool) # All are "accepted"
)

strong_learner = TorchLogistic(n_features=credit_data.features_count, seed_for_weight_init=1807)
# BASL related classes
basl_unbiaser = BASLPartialUnbiaser(
    filtering_quantiles={"lower" : 0.01, "upper" : 0.99},
    weak_learner=TorchLogistic(n_features=credit_data.features_count, seed_for_weight_init=187),
    strong_learner=strong_learner,
    holdout_percent=0.1,
    sampling_percent=0.8,
    label_bads_percent=0.1,
    label_goods_percent=0.1/2,
    max_iterations=5,
    isolation_forest=IsolationForest(n_estimators=100, max_samples="auto", random_state=1807),
    bayesian_metric = BayesianMetric(
        model=strong_learner,
        min_iterations=1e2,
        max_iterations=1e5,
        epsilon=1e-5,
        metric = batched_auroc,
        device=credit_data.device
    )
)

classifier_accepts = TorchLogistic(n_features=credit_data.features_count, seed_for_weight_init=781)
classifier_oracle = TorchLogistic(n_features=credit_data.features_count, seed_for_weight_init=10807)


TypeError: BASLPartialUnbiaser.__init__() got an unexpected keyword argument 'early_stop'

In [ ]:
classifier_accepts.lin_estimator.

{'filtering_quantiles': {'lower': 0.01, 'upper': 0.99},
 'weak_learner': TorchLogistic(
   (lin_estimator): Linear(in_features=2, out_features=1, bias=True)
 ),
 'strong_learner': TorchLogistic(
   (lin_estimator): Linear(in_features=2, out_features=1, bias=True)
 ),
 'holdout_percent': 0.1,
 'sampling_percent': 0.8,
 'label_bads_percent': 0.1,
 'label_goods_percent': 0.05,
 'max_iterations': 5,
 'early_stop': True,
 'isolation_forest': IsolationForest(random_state=1807),
 'bayesian_metric': <berebasl.estimation.bayesian_evaluation.BayesianMetric at 0x1f339197380>}

In [ ]:
if True:
    # Acceptance Loop

    stats: List[Dict[str, Union[float, int]]] = []
    models_state_dicts: List[Dict[str, Union[Dict[str, Any], str]]] = []

    for gen_nr in range(1, num_gens + 1):
        if gen_nr % 10 == 0:
            print("-- Iteration", f"{gen_nr}/{num_gens}:", credit_data.accepted_count, 
                "accepts and", credit_data.rejected_count, " rejects")
            
        ## Gather current statistics
        current_stats : dict = credit_data.data_stats()


        ## Get leakage-free data
        current_sample: CreditDataSample = credit_data.to_sample_dataset() # Ensure leakage-free data
        current_sample.manual_seed(initial_seed + gen_nr + 1) # internal rng handles seeds for splitting

        ## Accepts based scorecard
        ### Reset params to ensure no effect of last calculation
        classifier_accepts.reset_parameters_to_initial()
        classifier_accepts.fit(current_sample.features_labeled, current_sample.labels)

        classifier_accepts.eval()
        holdout_probs_bad_accepts_based = classifier_accepts.predict_proba(holdout_data.features)[..., 1]
        current_stats["auc_accepts"] = batched_auroc(holdout_probs_bad_accepts_based, holdout_data.default_flag).item()

        ## Oracle scorecard
        classifier_oracle.reset_parameters_to_initial()
        classifier_oracle.fit(credit_data.features, credit_data.default_flag) # Using explicitly all data

        classifier_oracle.eval()
        holdout_probs_bad_oracle = classifier_oracle.predict_proba(holdout_data.features)[..., 1]
        current_stats["auc_biased"] = batched_auroc(holdout_probs_bad_oracle, holdout_data.default_flag).item()

        ## Corrected scorecard
        augmented_sample: CreditDataSample = basl_unbiaser.basl_augment_sample(
            data=current_sample, 
            leave_orig_sample_untouched=True, 
            early_stop=True
        )
        basl_unbiaser.refit_model(which_one='strong', features=augmented_sample.features_labeled, labels=augmented_sample.labels)
        holdout_probs_basl = basl_unbiaser.predict_proba_model(which_one='strong', features=holdout_data.features)
        current_stats["auc_basl"] = batched_auroc(holdout_probs_basl, holdout_data.default_flag).item()

        stats.append(current_stats)

        if gen_nr == 1 or gen_nr % 10 == 0 or gen_nr == num_gens:
            models_state_dicts.append({
                "gen_round" : gen_nr,
                "accepts_model" : classifier_accepts.to_state_dict(),
                "oracle_model" : classifier_oracle.to_state_dict(),
                "basl_strong_model" : basl_unbiaser.strong_learner.to_state_dict()
            })

        if gen_nr < num_gens:
            ## Generate new data
            data_gen.manual_seed(initial_seed + gen_nr)
            # Let S:=sample_size
            feats_new_applicants, def_flag_new_applicants = data_gen.sample(sample_size, determinstic_mixture_weights) # [S, F], [S]
            with torch.no_grad():
                new_applicants_pred_def_probs = classifier_accepts.predict_proba(feats_new_applicants)[..., 1] # [S]
                new_applicants_accepted = new_applicants_pred_def_probs >= new_applicants_pred_def_probs.quantile(1-top_percent) # [S]

            max_accepts_allowed = round(sample_size*top_percent)
            currently_accepted = new_applicants_accepted.sum()
            if currently_accepted > max_accepts_allowed:
                count_to_flip = currently_accepted - max_accepts_allowed
                idx_accepted = torch.where(new_applicants_accepted)[0] # [currently_accepted,] 
                perm = torch.randperm(currently_accepted, generator=data_gen.rng, device=data_gen.device) 
                idx_to_flip = idx_accepted[perm[:count_to_flip]]
                new_applicants_accepted[idx_to_flip] = False
                
            credit_data.add_gen(feats_new_applicants, def_flag_new_applicants, new_applicants_accepted)
            

        break


In [53]:
stats

[{'sample_size': 200,
  'accept_ratio': 0.2,
  'bad_ratio_accepts': 0.1,
  'bad_ratio_rejects': 0.6,
  'bad_ratio_unbiased': 0.5,
  'auc_accepts': 0.8943493333333333,
  'auc_biased': 0.9353426666666668,
  'auc_basl': 0.9131488888888888}]